In [21]:
# ============================================================
# STEP 1 - ROSE DATASET INSPECTION
# Project: Neuro-Symbolic AI for Rose Disease Detection
# ============================================================

import os
from pathlib import Path
from collections import Counter

BASE_DIR = Path("/kaggle/input")

print("=" * 60)
print("ROSE DISEASE DATASET INSPECTION")
print("=" * 60)

# ------------------------------------------------------------
# 1. Find uploaded datasets
# ------------------------------------------------------------

print("\nDATASETS AVAILABLE IN KAGGLE:\n")

for item in BASE_DIR.iterdir():
    print("📁", item)

# ------------------------------------------------------------
# 2. Find all image files
# ------------------------------------------------------------

image_extensions = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".JPG", ".JPEG", ".PNG"
}

image_files = []

for root, dirs, files in os.walk(BASE_DIR):
    for file in files:
        if Path(file).suffix in image_extensions:
            image_files.append(Path(root) / file)

print("\n" + "=" * 60)
print("IMAGE INFORMATION")
print("=" * 60)

print("Total images found:", len(image_files))

# ------------------------------------------------------------
# 3. Show folder structure containing images
# ------------------------------------------------------------

folder_counts = Counter()

for img in image_files:
    folder_counts[img.parent] += 1

print("\nIMAGE COUNTS BY FOLDER:\n")

for folder, count in folder_counts.most_common():
    print(f"{count:5d} images  ->  {folder}")

# ------------------------------------------------------------
# 4. Show first 20 image paths
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("SAMPLE IMAGE PATHS")
print("=" * 60)

for img in image_files[:20]:
    print(img)

# ------------------------------------------------------------
# 5. Check whether dataset looks like rose/flower data
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("DATASET CHECK")
print("=" * 60)

rose_words = [
    "rose", "flower", "disease", "healthy",
    "black", "mildew", "spot", "fungal"
]

matches = []

for img in image_files:
    path_text = str(img).lower()

    if any(word in path_text for word in rose_words):
        matches.append(img)

print("Rose/disease-related image paths:", len(matches))

if len(image_files) == 0:
    print("\n❌ NO IMAGES FOUND")
    print("Please check whether the dataset was added correctly.")

elif len(matches) > 0:
    print("\n✅ Dataset contains rose/disease-related folders or filenames.")

else:
    print("\n⚠️ Images found, but folder names do not clearly identify rose diseases.")

print("\nSTEP 1 COMPLETED")

ROSE DISEASE DATASET INSPECTION

DATASETS AVAILABLE IN KAGGLE:

📁 /kaggle/input/datasets

IMAGE INFORMATION
Total images found: 2529

IMAGE COUNTS BY FOLDER:

  571 images  ->  /kaggle/input/datasets/jayanthmanjunath/original-rose-data-set/Original_Rose_leaf_Disease_Dataset/Original_Rose_leaf_Disease_Dataset/Train/Dry Leaf
  534 images  ->  /kaggle/input/datasets/jayanthmanjunath/original-rose-data-set/Original_Rose_leaf_Disease_Dataset/Original_Rose_leaf_Disease_Dataset/Train/Healthy Leaf
  342 images  ->  /kaggle/input/datasets/jayanthmanjunath/original-rose-data-set/Original_Rose_leaf_Disease_Dataset/Original_Rose_leaf_Disease_Dataset/Train/Insect Hole
  268 images  ->  /kaggle/input/datasets/jayanthmanjunath/original-rose-data-set/Original_Rose_leaf_Disease_Dataset/Original_Rose_leaf_Disease_Dataset/Train/Black Spot
  252 images  ->  /kaggle/input/datasets/jayanthmanjunath/original-rose-data-set/Original_Rose_leaf_Disease_Dataset/Original_Rose_leaf_Disease_Dataset/Train/Downy Milde

In [22]:
# ============================================================
# FAST KAGGLE TEST VERSION
# NEURO-SYMBOLIC AI FOR ROSE DISEASE DETECTION
# ============================================================

import os
import json
import shutil
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# 1. SETTINGS
# ============================================================

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

# FAST TEST RUN
EPOCHS = 5

# Vision Transformer parameters
PATCH_SIZE = 16
PROJECTION_DIM = 256
NUM_HEADS = 4
TRANSFORMER_DEPTH = 4
MLP_DIM = 512
DROPOUT = 0.10

random.seed(SEED)
np.random.seed(SEED)

print("=" * 65)
print("FAST NEURO-SYMBOLIC ROSE DISEASE AI")
print("=" * 65)


# ============================================================
# 2. GPU CHECK
# ============================================================

import tensorflow as tf

tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("\n✅ GPU AVAILABLE")
    print(gpus)
else:
    print("\n⚠️ GPU NOT AVAILABLE")


# ============================================================
# 3. FIND DATASET
# ============================================================

KAGGLE_INPUT = Path("/kaggle/input")

dataset_root = None

possible_paths = [

    Path(
        "/kaggle/input/datasets/jayanthmanjunath/"
        "original-rose-data-set/"
        "Original_Rose_leaf_Disease_Dataset/"
        "Original_Rose_leaf_Disease_Dataset"
    ),

    Path(
        "/kaggle/input/original-rose-data-set/"
        "Original_Rose_leaf_Disease_Dataset/"
        "Original_Rose_leaf_Disease_Dataset"
    ),

    Path(
        "/kaggle/input/original-rose-dataset/"
        "Original_Rose_leaf_Disease_Dataset/"
        "Original_Rose_leaf_Disease_Dataset"
    )
]

for path in possible_paths:

    if path.exists():

        dataset_root = path
        break


# Automatic fallback search

if dataset_root is None:

    for root, dirs, files in os.walk(KAGGLE_INPUT):

        root_path = Path(root)

        if (
            root_path.name == "Train"
            and (root_path / "Dry Leaf").exists()
        ):

            dataset_root = root_path.parent
            break


if dataset_root is None:

    raise FileNotFoundError(
        "❌ Rose dataset not found. "
        "Please attach the Kaggle dataset."
    )


print("\n✅ Dataset found:")
print(dataset_root)


# ============================================================
# 4. TRAIN / TEST PATHS
# ============================================================

train_path = dataset_root / "Train"
test_path = dataset_root / "Test"


# ============================================================
# 5. DATASET CLASSES
# ============================================================

CLASS_MAPPING = {

    "Black Spot": "Black Spot",

    # Dataset naming correction
    "Black Sport": "Black Spot",

    "Downy Mildew": "Downy Mildew",

    "Dry Leaf": "Dry Leaf",

    "Healthy Leaf": "Healthy Leaf",

    "Insect Hole": "Insect Hole"
}

CLASS_NAMES = [

    "Black Spot",

    "Downy Mildew",

    "Dry Leaf",

    "Healthy Leaf",

    "Insect Hole"
]

NUM_CLASSES = len(CLASS_NAMES)


print("\nFinal classes:")

for i, c in enumerate(CLASS_NAMES):

    print(i, "->", c)


# ============================================================
# 6. COLLECT LABELLED IMAGES
# ============================================================

extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

records = []

for split in [train_path, test_path]:

    if not split.exists():
        continue

    for root, dirs, files in os.walk(split):

        folder = Path(root).name

        if folder not in CLASS_MAPPING:
            continue

        final_class = CLASS_MAPPING[folder]

        for file in files:

            if Path(file).suffix.lower() in extensions:

                records.append({

                    "filepath":
                        str(Path(root) / file),

                    "class":
                        final_class

                })


df = pd.DataFrame(records)

print(
    "\nTotal labelled images:",
    len(df)
)


# ============================================================
# 7. CLASS DISTRIBUTION
# ============================================================

print("\nClass distribution:")

for c in CLASS_NAMES:

    print(
        f"{c:20s}:",
        int(
            (df["class"] == c).sum()
        )
    )


# ============================================================
# 8. STRATIFIED SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(

    df,

    test_size=0.20,

    stratify=df["class"],

    random_state=SEED
)

val_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["class"],

    random_state=SEED
)


print("\nDataset split:")

print("Train      :", len(train_df))

print("Validation :", len(val_df))

print("Test       :", len(test_df))


# ============================================================
# 9. CREATE FAST DATASET
# ============================================================
#
# IMPORTANT:
# We DO NOT copy 2,458 images.
#
# TensorFlow loads the original images directly.
#
# This makes the notebook much faster.
# ============================================================

def dataframe_to_dataset(
    dataframe,
    shuffle=False
):

    paths = dataframe["filepath"].values

    labels = np.array([

        CLASS_NAMES.index(c)

        for c in dataframe["class"]

    ])

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels)
    )


    def load_image(path, label):

        image = tf.io.read_file(path)

        image = tf.image.decode_image(
            image,
            channels=3,
            expand_animations=False
        )

        image = tf.image.resize(
            image,
            IMAGE_SIZE
        )

        image = tf.cast(
            image,
            tf.float32
        )

        return image, label


    dataset = dataset.map(

        load_image,

        num_parallel_calls=tf.data.AUTOTUNE

    )


    if shuffle:

        dataset = dataset.shuffle(
            1000,
            seed=SEED
        )


    dataset = dataset.batch(
        BATCH_SIZE
    )

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset


print("\nLoading TensorFlow datasets...")

train_ds = dataframe_to_dataset(
    train_df,
    shuffle=True
)

val_ds = dataframe_to_dataset(
    val_df
)

test_ds = dataframe_to_dataset(
    test_df
)

print("✅ Datasets ready")


# ============================================================
# 10. DATA AUGMENTATION
# ============================================================

augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip(
        "horizontal"
    ),

    tf.keras.layers.RandomRotation(
        0.05
    ),

    tf.keras.layers.RandomZoom(
        0.08
    )

])


# ============================================================
# 11. PATCH EXTRACTION
# ============================================================

class Patches(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        patch_size
    ):

        super().__init__()

        self.patch_size = patch_size


    def call(self, images):

        batch_size = tf.shape(images)[0]

        patches = tf.image.extract_patches(

            images=images,

            sizes=[
                1,
                self.patch_size,
                self.patch_size,
                1
            ],

            strides=[
                1,
                self.patch_size,
                self.patch_size,
                1
            ],

            rates=[
                1,
                1,
                1,
                1
            ],

            padding="VALID"

        )

        patch_dims = patches.shape[-1]

        patches = tf.reshape(

            patches,

            [
                batch_size,
                -1,
                patch_dims
            ]

        )

        return patches


# ============================================================
# 12. PATCH ENCODER
# ============================================================

class PatchEncoder(
    tf.keras.layers.Layer
):

    def __init__(
        self,
        num_patches,
        projection_dim
    ):

        super().__init__()

        self.projection = (
            tf.keras.layers.Dense(
                projection_dim
            )
        )

        self.position_embedding = (

            tf.keras.layers.Embedding(

                input_dim=num_patches,

                output_dim=projection_dim

            )

        )

        self.num_patches = num_patches


    def call(self, patches):

        positions = tf.range(

            start=0,

            limit=self.num_patches,

            delta=1

        )

        encoded = (

            self.projection(patches)

            +

            self.position_embedding(
                positions
            )

        )

        return encoded


# ============================================================
# 13. TRANSFORMER BLOCK
# ============================================================

def mlp_block(
    x,
    hidden_units,
    dropout
):

    for units in hidden_units:

        x = tf.keras.layers.Dense(

            units,

            activation=tf.nn.gelu

        )(x)

        x = tf.keras.layers.Dropout(
            dropout
        )(x)

    return x


def transformer_block(x):

    x1 = tf.keras.layers.LayerNormalization(
        epsilon=1e-6
    )(x)


    attention = (

        tf.keras.layers.MultiHeadAttention(

            num_heads=NUM_HEADS,

            key_dim=
                PROJECTION_DIM // NUM_HEADS,

            dropout=DROPOUT

        )

        (x1, x1)

    )


    x2 = tf.keras.layers.Add()(
        [attention, x]
    )


    x3 = tf.keras.layers.LayerNormalization(
        epsilon=1e-6
    )(x2)


    x3 = mlp_block(

        x3,

        [
            MLP_DIM,
            PROJECTION_DIM
        ],

        DROPOUT

    )


    return tf.keras.layers.Add()(
        [x3, x2]
    )


# ============================================================
# 14. BUILD FAST ViT
# ============================================================

NUM_PATCHES = (
    224 // PATCH_SIZE
) ** 2


inputs = tf.keras.Input(
    shape=(224, 224, 3)
)


x = augmentation(inputs)


x = tf.keras.layers.Rescaling(
    1.0 / 255
)(x)


x = Patches(
    PATCH_SIZE
)(x)


x = PatchEncoder(

    NUM_PATCHES,

    PROJECTION_DIM

)(x)


for _ in range(
    TRANSFORMER_DEPTH
):

    x = transformer_block(x)


x = tf.keras.layers.LayerNormalization(
    epsilon=1e-6
)(x)


x = tf.keras.layers.GlobalAveragePooling1D()(
    x
)


x = tf.keras.layers.Dropout(
    0.30
)(x)


x = tf.keras.layers.Dense(
    128,
    activation="gelu"
)(x)


x = tf.keras.layers.Dropout(
    0.20
)(x)


outputs = tf.keras.layers.Dense(

    NUM_CLASSES,

    activation="softmax"

)(x)


model = tf.keras.Model(

    inputs,

    outputs,

    name="Fast_Rose_Vision_Transformer"

)


# ============================================================
# 15. MODEL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("MODEL CREATED")
print("=" * 65)

model.summary()


# ============================================================
# 16. COMPILE
# ============================================================

model.compile(

    optimizer=tf.keras.optimizers.AdamW(

        learning_rate=1e-4,

        weight_decay=1e-4

    ),

    loss=
        "sparse_categorical_crossentropy",

    metrics=[
        "accuracy"
    ]

)


# ============================================================
# 17. CALLBACKS
# ============================================================

MODEL_PATH = (
    "/kaggle/working/"
    "rose_vit_fast.keras"
)


callbacks = [

    tf.keras.callbacks.ModelCheckpoint(

        MODEL_PATH,

        monitor="val_accuracy",

        save_best_only=True,

        mode="max",

        verbose=1

    ),

    tf.keras.callbacks.EarlyStopping(

        monitor="val_accuracy",

        patience=3,

        restore_best_weights=True,

        mode="max",

        verbose=1

    )

]


# ============================================================
# 18. TRAIN
# ============================================================

print("\n" + "=" * 65)

print(
    f"STARTING FAST TRAINING — {EPOCHS} EPOCHS"
)

print("=" * 65)


history = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=EPOCHS,

    callbacks=callbacks

)


# ============================================================
# 19. TEST MODEL
# ============================================================

print("\n" + "=" * 65)
print("TESTING MODEL")
print("=" * 65)


test_loss, test_accuracy = model.evaluate(

    test_ds,

    verbose=1

)


print(

    "\n✅ TEST ACCURACY:",

    round(
        test_accuracy * 100,
        2
    ),

    "%"

)


# ============================================================
# 20. CONFUSION MATRIX
# ============================================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report
)


y_true = []

y_pred = []


for images, labels in test_ds:

    predictions = model.predict(
        images,
        verbose=0
    )

    y_true.extend(
        labels.numpy()
    )

    y_pred.extend(
        np.argmax(
            predictions,
            axis=1
        )
    )


print("\n" + "=" * 65)
print("CLASSIFICATION REPORT")
print("=" * 65)


print(

    classification_report(

        y_true,

        y_pred,

        target_names=CLASS_NAMES,

        digits=4

    )

)


cm = confusion_matrix(
    y_true,
    y_pred
)


plt.figure(
    figsize=(8, 6)
)

plt.imshow(
    cm,
    interpolation="nearest"
)

plt.title(
    "Rose Disease Confusion Matrix"
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "Actual Class"
)

plt.colorbar()

plt.xticks(
    range(NUM_CLASSES),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(NUM_CLASSES),
    CLASS_NAMES
)


for i in range(NUM_CLASSES):

    for j in range(NUM_CLASSES):

        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )


plt.tight_layout()

plt.show()


# ============================================================
# 21. SAVE CLASS NAMES
# ============================================================

CLASS_PATH = (
    "/kaggle/working/"
    "rose_class_names.json"
)


with open(
    CLASS_PATH,
    "w"
) as f:

    json.dump(
        CLASS_NAMES,
        f,
        indent=4
    )


# ============================================================
# 22. SINGLE IMAGE PREDICTION
# ============================================================

def predict_rose_disease(
    image_path
):

    image = tf.keras.utils.load_img(

        image_path,

        target_size=IMAGE_SIZE

    )


    image = tf.keras.utils.img_to_array(
        image
    )


    image = np.expand_dims(
        image,
        axis=0
    )


    probabilities = model.predict(

        image,

        verbose=0

    )[0]


    index = np.argmax(
        probabilities
    )


    return {

        "disease":
            CLASS_NAMES[index],

        "confidence":
            float(
                probabilities[index] * 100
            ),

        "probabilities": {

            CLASS_NAMES[i]:
                float(probabilities[i])

            for i in range(NUM_CLASSES)

        }

    }


# ============================================================
# 23. NEURO-SYMBOLIC RULE ENGINE
# ============================================================

def environmental_risk(

    temperature,

    humidity,

    leaf_wetness,

    soil_moisture=None

):

    score = 0

    rules = []


    # Humidity rule

    if humidity >= 85:

        score += 30

        rules.append(
            "High humidity"
        )

    elif humidity >= 75:

        score += 20

        rules.append(
            "Moderately high humidity"
        )


    # Temperature rule

    if 18 <= temperature <= 28:

        score += 25

        rules.append(
            "Favorable temperature range"
        )


    # Leaf wetness rule

    if leaf_wetness >= 80:

        score += 35

        rules.append(
            "High leaf wetness"
        )

    elif leaf_wetness >= 60:

        score += 20

        rules.append(
            "Moderate leaf wetness"
        )


    # Soil moisture

    if (

        soil_moisture is not None

        and

        soil_moisture >= 80

    ):

        score += 10

        rules.append(
            "Very high soil moisture"
        )


    score = min(
        score,
        100
    )


    if score >= 70:

        level = "HIGH"

    elif score >= 40:

        level = "MEDIUM"

    else:

        level = "LOW"


    return {

        "risk_score":
            score,

        "risk_level":
            level,

        "rules":
            rules

    }


# ============================================================
# 24. NEURO-SYMBOLIC FUSION
# ============================================================

def neuro_symbolic_analysis(

    image_path,

    temperature,

    humidity,

    leaf_wetness,

    soil_moisture=None

):

    image_result = predict_rose_disease(
        image_path
    )


    environment = environmental_risk(

        temperature,

        humidity,

        leaf_wetness,

        soil_moisture

    )


    image_confidence = (
        image_result["confidence"]
    )


    environmental_score = (
        environment["risk_score"]
    )


    # Neural + symbolic fusion

    fused_score = (

        0.60 *
        image_confidence

        +

        0.40 *
        environmental_score

    )


    if fused_score >= 75:

        final_risk = "HIGH"

    elif fused_score >= 45:

        final_risk = "MEDIUM"

    else:

        final_risk = "LOW"


    return {

        "disease":
            image_result["disease"],

        "image_confidence":
            image_confidence,

        "environmental_score":
            environmental_score,

        "final_risk_score":
            fused_score,

        "final_risk":
            final_risk,

        "rules":
            environment["rules"]

    }


# ============================================================
# 25. MANAGEMENT RECOMMENDATION
# ============================================================

def recommendation(
    disease,
    risk
):

    if disease == "Black Spot":

        actions = [

            "Remove severely affected leaves",

            "Improve air circulation",

            "Avoid prolonged leaf wetness",

            "Increase monitoring",

            "Consider a locally registered fungicide "
            "appropriate for rose black spot and follow "
            "the product label and local agricultural guidance."

        ]


    elif disease == "Downy Mildew":

        actions = [

            "Remove heavily affected material",

            "Improve ventilation",

            "Reduce prolonged leaf wetness",

            "Avoid unnecessary overhead irrigation",

            "Use locally approved disease-management "
            "measures according to agricultural guidance."

        ]


    elif disease == "Dry Leaf":

        actions = [

            "Check irrigation",

            "Monitor soil moisture",

            "Inspect environmental stress",

            "Remove severely damaged leaves"

        ]


    elif disease == "Healthy Leaf":

        actions = [

            "Continue regular monitoring",

            "Maintain good ventilation",

            "Maintain suitable irrigation",

            "Continue preventive monitoring"

        ]


    else:

        actions = [

            "Inspect the plant for insect activity",

            "Remove severely damaged leaves",

            "Continue monitoring",

            "Use an appropriate locally approved "
            "control method if required"

        ]


    if risk == "HIGH":

        actions.insert(

            0,

            "⚠️ HIGH RISK — take preventive action "
            "and increase monitoring frequency."

        )


    elif risk == "MEDIUM":

        actions.insert(

            0,

            "⚠️ MEDIUM RISK — continue close monitoring."

        )


    return actions


# ============================================================
# 26. IoT SENSOR PROCESSING
# ============================================================

def process_iot_data(
    temperature,
    humidity,
    leaf_wetness,
    soil_moisture=None
):

    return environmental_risk(

        temperature,

        humidity,

        leaf_wetness,

        soil_moisture

    )


# ============================================================
# 27. IoT DEMO
# ============================================================

iot = process_iot_data(

    temperature=24.5,

    humidity=87,

    leaf_wetness=81,

    soil_moisture=68

)


print("\n" + "=" * 65)
print("REAL-TIME IoT DEMONSTRATION")
print("=" * 65)

print(
    "Temperature : 24.5 °C"
)

print(
    "Humidity    : 87 %"
)

print(
    "Leaf Wetness: 81 %"
)

print(
    "Soil Moisture: 68 %"
)

print(
    "Risk Score  :",
    iot["risk_score"]
)

print(
    "Risk Level  :",
    iot["risk_level"]
)

print("\nRules triggered:")

for rule in iot["rules"]:

    print(
        " •",
        rule
    )


# ============================================================
# 28. SAVE PROJECT INFORMATION
# ============================================================

project_info = {

    "project":
        "Neuro-Symbolic AI Framework for "
        "Pre-Symptomatic Detection of Fungal Diseases "
        "in Commercial Floriculture Using Real-Time IoT Sensor Data",

    "model":
        "Vision Transformer",

    "classes":
        CLASS_NAMES,

    "image_size":
        "224x224",

    "patch_size":
        PATCH_SIZE,

    "embedding_dimension":
        PROJECTION_DIM,

    "transformer_depth":
        TRANSFORMER_DEPTH,

    "attention_heads":
        NUM_HEADS,

    "mlp_dimension":
        MLP_DIM,

    "training_epochs":
        EPOCHS,

    "neural_component":
        "Vision Transformer",

    "symbolic_component":
        "Environmental risk rules",

    "sensor_inputs": [

        "Temperature",

        "Humidity",

        "Leaf Wetness",

        "Soil Moisture"

    ],

    "fusion":
        "60% neural image evidence + "
        "40% symbolic environmental risk",

    "test_accuracy":
        float(test_accuracy)

}


with open(

    "/kaggle/working/"
    "project_information.json",

    "w"

) as f:

    json.dump(
        project_info,
        f,
        indent=4
    )


# ============================================================
# 29. FINAL OUTPUT
# ============================================================

print("\n\n" + "=" * 65)
print("✅ FAST TEST PIPELINE COMPLETED")
print("=" * 65)

print(
    "\nTest Accuracy:",
    round(
        test_accuracy * 100,
        2
    ),
    "%"
)

print(
    "\nModel saved at:"
)

print(
    MODEL_PATH
)

print(
    "\nClass names saved at:"
)

print(
    CLASS_PATH
)

print(
    "\nProject information saved at:"
)

print(
    "/kaggle/working/project_information.json"
)

print("\nNext stage:")

print(
    "ESP32 → IoT Sensors → FastAPI → "
    "Database → Neuro-Symbolic AI → Dashboard"
)

print("=" * 65)

FAST NEURO-SYMBOLIC ROSE DISEASE AI

⚠️ GPU NOT AVAILABLE

✅ Dataset found:
/kaggle/input/datasets/jayanthmanjunath/original-rose-data-set/Original_Rose_leaf_Disease_Dataset/Original_Rose_leaf_Disease_Dataset

Final classes:
0 -> Black Spot
1 -> Downy Mildew
2 -> Dry Leaf
3 -> Healthy Leaf
4 -> Insect Hole

Total labelled images: 2458

Class distribution:
Black Spot          : 335
Downy Mildew        : 316
Dry Leaf            : 712
Healthy Leaf        : 668
Insect Hole         : 427

Dataset split:
Train      : 1966
Validation : 246
Test       : 246

Loading TensorFlow datasets...
✅ Datasets ready

MODEL CREATED


Model: "Fast_Rose_Vision_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_1        │ (None, 224, 224,  │          0 │ input_layer_4[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ sequential_1[0][… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patches_3 (Patches) │ (None, None, 768) │          0 │ rescaling_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_encoder_3     │ (None, 196, 256)  │    247,040 │ patches_3[0][0]   │
│ (PatchEncoder)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 256)  │        512 │ patch_encoder_3[… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 196, 256)  │    263,168 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_20 (Add)        │ (None, 196, 256)  │          0 │ multi_head_atten… │
│                     │                   │            │ patch_encoder_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 256)  │        512 │ add_20[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_28 (Dense)    │ (None, 196, 512)  │    131,584 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_39          │ (None, 196, 512)  │          0 │ dense_28[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_29 (Dense)    │ (None, 196, 256)  │    131,328 │ dropout_39[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_40          │ (None, 196, 256)  │          0 │ dense_29[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_21 (Add)        │ (None, 196, 256)  │          0 │ dropout_40[0][0], │
│                     │                   │            │ add_20[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 256)  │        512 │ add_21[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 196, 256)  │    263,168 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_22 (Add)        │ (None, 196, 256)  │          0 │ multi_head_atten… │
│                     │                   │            │ add_21[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 256)  │        512 │ add_22[0][0]    

 Total params: 2,389,509 (9.12 MB)

 Trainable params: 2,389,509 (9.12 MB)

 Non-trainable params: 0 (0.00 B)


STARTING FAST TRAINING — 5 EPOCHS
Epoch 1/5
50/62 ━━━━━━━━━━━━━━━━━━━━ 25s 2s/step - accuracy: 0.2891 - loss: 1.6610

KeyboardInterrupt: 

In [ ]:
# ============================================================
# STEP 2: INSPECT ROSE/FLOWER ARCHIVE IMAGES
# ============================================================

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import random

print("=" * 60)
print("STEP 2: ROSE/FLOWER IMAGE INSPECTION")
print("=" * 60)

# Locate archive images automatically
archive_paths = list(Path("/kaggle/input").rglob("archive/archive/*.jpg"))
archive_paths += list(Path("/kaggle/input").rglob("archive/archive/*.jpeg"))
archive_paths += list(Path("/kaggle/input").rglob("archive/archive/*.png"))

# Remove duplicates
archive_paths = list(dict.fromkeys(archive_paths))

print(f"\nFlower/archive images found: {len(archive_paths)}")

if len(archive_paths) == 0:
    print("\n❌ No archive images found.")
else:
    print("\nSample filenames:")
    for p in archive_paths[:20]:
        print("  ", p.name)

    # Select images to display
    sample_count = min(20, len(archive_paths))
    samples = random.sample(archive_paths, sample_count)

    plt.figure(figsize=(16, 12))

    for i, img_path in enumerate(samples):
        try:
            img = Image.open(img_path).convert("RGB")

            ax = plt.subplot(4, 5, i + 1)
            ax.imshow(img)
            ax.set_title(img_path.name[:22], fontsize=8)
            ax.axis("off")

        except Exception as e:
            print("Could not read:", img_path, e)

    plt.suptitle(
        "Rose / Flower Archive Images",
        fontsize=18
    )

    plt.tight_layout()
    plt.show()

print("\n" + "=" * 60)
print("INSPECTION COMPLETED")
print("=" * 60)

In [ ]:
# ============================================================
# STEP 3: VISUAL INSPECTION OF ALL ROSE/FLOWER IMAGES
# ============================================================

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import math

print("=" * 60)
print("STEP 3: ALL ROSE/FLOWER IMAGES")
print("=" * 60)

# Find archive images
archive_paths = list(Path("/kaggle/input").rglob("archive/archive/*.jpg"))
archive_paths += list(Path("/kaggle/input").rglob("archive/archive/*.jpeg"))
archive_paths += list(Path("/kaggle/input").rglob("archive/archive/*.png"))

archive_paths = list(dict.fromkeys(archive_paths))

print(f"\nTotal flower images: {len(archive_paths)}")

if len(archive_paths) > 0:

    # Sort for easier inspection
    archive_paths = sorted(archive_paths, key=lambda x: x.name.lower())

    cols = 6
    rows = math.ceil(len(archive_paths) / cols)

    plt.figure(figsize=(18, rows * 3))

    for i, img_path in enumerate(archive_paths):

        try:
            img = Image.open(img_path).convert("RGB")

            ax = plt.subplot(rows, cols, i + 1)
            ax.imshow(img)
            ax.set_title(img_path.name[:25], fontsize=7)
            ax.axis("off")

        except Exception as e:
            print("Error:", img_path.name, e)

    plt.suptitle(
        "Rose / Flower Image Dataset - Complete Inspection",
        fontsize=18
    )

    plt.tight_layout()
    plt.show()

print("\n" + "=" * 60)
print("STEP 3 COMPLETED")
print("=" * 60)

In [ ]:
# ======================================================================
# NEURO-SYMBOLIC ROSE DISEASE AI
# FIXED NEXT-STAGE PIPELINE
# ======================================================================

import os
import json
import sqlite3
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from PIL import Image

print("=" * 70)
print("NEURO-SYMBOLIC ROSE DISEASE AI")
print("FIXED NEXT-STAGE PIPELINE")
print("=" * 70)

print("\nTensorFlow:", tf.__version__)
print("GPU:", "AVAILABLE" if tf.config.list_physical_devices("GPU") else "CPU MODE")


# ======================================================================
# 1. MODEL PATH
# ======================================================================

MODEL_PATH = "/kaggle/working/rose_vit_fast.keras"
CLASS_PATH = "/kaggle/working/rose_class_names.json"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        "Model not found: " + MODEL_PATH
    )

print("\nModel found:")
print(MODEL_PATH)


# ======================================================================
# 2. DISEASE CLASSES
# ======================================================================

class_names = [
    "Black Spot",
    "Downy Mildew",
    "Dry Leaf",
    "Healthy Leaf",
    "Insect Hole"
]

if os.path.exists(CLASS_PATH):
    try:
        with open(CLASS_PATH, "r") as f:
            saved_classes = json.load(f)

        if isinstance(saved_classes, dict):
            try:
                class_names = [
                    saved_classes[str(i)]
                    for i in range(len(saved_classes))
                ]
            except:
                pass

        elif isinstance(saved_classes, list):
            class_names = saved_classes

    except Exception as e:
        print("Class file could not be read. Using default classes.")

print("\nDisease classes:")
for i, name in enumerate(class_names):
    print(i, "->", name)


# ======================================================================
# 3. CUSTOM VISION TRANSFORMER LAYERS
# ======================================================================
# IMPORTANT:
# These classes are required to reload the saved ViT model.
# ======================================================================

@tf.keras.utils.register_keras_serializable()
class Patches(tf.keras.layers.Layer):

    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):

        batch_size = tf.shape(images)[0]

        patches = tf.image.extract_patches(
            images=images,
            sizes=[
                1,
                self.patch_size,
                self.patch_size,
                1
            ],
            strides=[
                1,
                self.patch_size,
                self.patch_size,
                1
            ],
            rates=[1, 1, 1, 1],
            padding="VALID"
        )

        patch_dims = patches.shape[-1]

        patches = tf.reshape(
            patches,
            [
                batch_size,
                -1,
                patch_dims
            ]
        )

        return patches

    def get_config(self):

        config = super().get_config()

        config.update({
            "patch_size": self.patch_size
        })

        return config


@tf.keras.utils.register_keras_serializable()
class PatchEncoder(tf.keras.layers.Layer):

    def __init__(
        self,
        num_patches,
        projection_dim,
        **kwargs
    ):
        super().__init__(**kwargs)

        self.num_patches = num_patches
        self.projection_dim = projection_dim

        self.projection = tf.keras.layers.Dense(
            units=projection_dim
        )

        self.position_embedding = (
            tf.keras.layers.Embedding(
                input_dim=num_patches,
                output_dim=projection_dim
            )
        )

    def call(self, patches):

        positions = tf.range(
            start=0,
            limit=self.num_patches,
            delta=1
        )

        encoded = (
            self.projection(patches)
            +
            self.position_embedding(positions)
        )

        return encoded

    def get_config(self):

        config = super().get_config()

        config.update({
            "num_patches": self.num_patches,
            "projection_dim": self.projection_dim
        })

        return config


# ======================================================================
# 4. LOAD TRAINED VIT MODEL
# ======================================================================

print("\nLoading trained ViT model...")

try:

    model = tf.keras.models.load_model(
        MODEL_PATH,
        custom_objects={
            "Patches": Patches,
            "PatchEncoder": PatchEncoder
        },
        compile=False
    )

    print("✅ MODEL LOADED SUCCESSFULLY")

except Exception as e:

    print("\n❌ MODEL LOADING FAILED")
    print(str(e))

    raise


print("\nModel input:")
print(model.input_shape)

print("\nModel output:")
print(model.output_shape)

print("\nTotal parameters:")
print(model.count_params())


# ======================================================================
# 5. IMAGE PREPROCESSING
# ======================================================================

IMG_SIZE = 224

def preprocess_image(image_path):

    image = Image.open(image_path).convert("RGB")

    image = image.resize(
        (IMG_SIZE, IMG_SIZE)
    )

    image_array = np.array(
        image
    ).astype("float32") / 255.0

    image_array = np.expand_dims(
        image_array,
        axis=0
    )

    return image_array


# ======================================================================
# 6. DISEASE PREDICTION
# ======================================================================

def predict_disease(image_path):

    image = preprocess_image(
        image_path
    )

    predictions = model.predict(
        image,
        verbose=0
    )[0]

    predicted_index = int(
        np.argmax(predictions)
    )

    predicted_disease = class_names[
        predicted_index
    ]

    confidence = float(
        predictions[predicted_index]
    ) * 100

    return (
        predicted_disease,
        confidence,
        predictions
    )


# ======================================================================
# 7. NEURO-SYMBOLIC ENVIRONMENTAL RULE ENGINE
# ======================================================================

def environmental_risk(
    temperature,
    humidity,
    leaf_wetness,
    soil_moisture
):

    score = 0

    rules = []

    # --------------------------------------------------------------
    # Humidity rule
    # --------------------------------------------------------------

    if humidity >= 85:

        score += 30

        rules.append(
            "High humidity"
        )

    elif humidity >= 75:

        score += 20

        rules.append(
            "Moderately high humidity"
        )


    # --------------------------------------------------------------
    # Temperature rule
    # --------------------------------------------------------------

    if 18 <= temperature <= 28:

        score += 25

        rules.append(
            "Favorable temperature range"
        )


    # --------------------------------------------------------------
    # Leaf wetness rule
    # --------------------------------------------------------------

    if leaf_wetness >= 80:

        score += 35

        rules.append(
            "High leaf wetness"
        )

    elif leaf_wetness >= 60:

        score += 20

        rules.append(
            "Moderate leaf wetness"
        )


    # --------------------------------------------------------------
    # Soil moisture rule
    # --------------------------------------------------------------

    if soil_moisture >= 80:

        score += 10

        rules.append(
            "High soil moisture"
        )


    score = min(
        score,
        100
    )


    if score >= 70:

        level = "HIGH"

    elif score >= 40:

        level = "MEDIUM"

    else:

        level = "LOW"


    return score, level, rules


# ======================================================================
# 8. NEURO-SYMBOLIC FUSION
# ======================================================================

def neuro_symbolic_prediction(
    image_confidence,
    environmental_score
):

    final_risk = (
        0.60 * image_confidence
        +
        0.40 * environmental_score
    )

    final_risk = min(
        final_risk,
        100
    )

    if final_risk >= 70:

        level = "HIGH"

    elif final_risk >= 40:

        level = "MEDIUM"

    else:

        level = "LOW"

    return final_risk, level


# ======================================================================
# 9. MANAGEMENT RECOMMENDATION
# ======================================================================

def get_recommendation(
    disease,
    risk_level
):

    if risk_level == "HIGH":

        if disease == "Black Spot":

            return (
                "High disease risk. Remove severely affected leaves, "
                "improve air circulation, avoid prolonged leaf wetness, "
                "and consider a locally registered fungicide suitable "
                "for rose black spot. Follow the product label and "
                "local agricultural guidance."
            )

        elif disease == "Downy Mildew":

            return (
                "High disease risk. Reduce leaf wetness, improve "
                "ventilation, remove severely affected material, "
                "and consider an appropriate locally registered "
                "fungicide according to the product label."
            )

        elif disease == "Insect Hole":

            return (
                "Possible insect damage. Inspect plants for active "
                "pests and consider an appropriate locally approved "
                "pest-management approach."
            )

        elif disease == "Dry Leaf":

            return (
                "Dry-leaf condition detected. Check irrigation, "
                "temperature, root-zone moisture and plant stress."
            )

        else:

            return (
                "High environmental risk detected. Inspect plants "
                "closely and take preventive management action."
            )


    elif risk_level == "MEDIUM":

        return (
            "Moderate risk. Continue monitoring environmental "
            "conditions, improve ventilation and reduce prolonged "
            "leaf wetness where possible."
        )


    else:

        return (
            "Low current risk. Continue regular monitoring and "
            "maintain good greenhouse hygiene."
        )


# ======================================================================
# 10. OCCLUSION-BASED EXPLAINABILITY
# ======================================================================

def create_explanation(
    image_path,
    grid_size=7
):

    image = Image.open(
        image_path
    ).convert("RGB")

    image = image.resize(
        (IMG_SIZE, IMG_SIZE)
    )

    original = (
        np.array(image)
        .astype("float32")
        / 255.0
    )

    base = np.expand_dims(
        original,
        axis=0
    )

    base_prediction = model.predict(
        base,
        verbose=0
    )[0]

    target_class = np.argmax(
        base_prediction
    )

    base_score = base_prediction[
        target_class
    ]

    heatmap = np.zeros(
        (grid_size, grid_size),
        dtype=np.float32
    )

    patch_h = IMG_SIZE // grid_size
    patch_w = IMG_SIZE // grid_size

    for r in range(grid_size):

        for c in range(grid_size):

            modified = original.copy()

            y1 = r * patch_h
            y2 = (r + 1) * patch_h

            x1 = c * patch_w
            x2 = (c + 1) * patch_w

            modified[
                y1:y2,
                x1:x2,
                :
            ] = 0.5

            modified = np.expand_dims(
                modified,
                axis=0
            )

            prediction = model.predict(
                modified,
                verbose=0
            )[0]

            heatmap[r, c] = max(
                0,
                base_score - prediction[target_class]
            )


    if heatmap.max() > 0:

        heatmap = (
            heatmap
            /
            heatmap.max()
        )

    heatmap = tf.image.resize(
        heatmap[..., np.newaxis],
        (IMG_SIZE, IMG_SIZE)
    ).numpy()[:, :, 0]

    return (
        original,
        heatmap,
        class_names[target_class]
    )


# ======================================================================
# 11. COMPLETE ANALYSIS FUNCTION
# ======================================================================

def analyze_rose(
    image_path,
    temperature=24.5,
    humidity=87,
    leaf_wetness=81,
    soil_moisture=68
):

    print("\n")
    print("=" * 70)
    print("ROSE DISEASE ANALYSIS")
    print("=" * 70)

    # --------------------------------------------------------------
    # Neural prediction
    # --------------------------------------------------------------

    disease, confidence, probabilities = (
        predict_disease(
            image_path
        )
    )

    print("\nAI Vision Result:")
    print("Disease:", disease)
    print("Confidence:", round(confidence, 2), "%")


    # --------------------------------------------------------------
    # Symbolic reasoning
    # --------------------------------------------------------------

    env_score, env_level, rules = (
        environmental_risk(
            temperature,
            humidity,
            leaf_wetness,
            soil_moisture
        )
    )

    print("\nEnvironmental Rules:")

    print(
        "Temperature:",
        temperature,
        "°C"
    )

    print(
        "Humidity:",
        humidity,
        "%"
    )

    print(
        "Leaf Wetness:",
        leaf_wetness,
        "%"
    )

    print(
        "Soil Moisture:",
        soil_moisture,
        "%"
    )

    print(
        "\nEnvironmental Risk:",
        env_score,
        "%",
        "(",
        env_level,
        ")"
    )

    print("\nTriggered Rules:")

    if rules:

        for rule in rules:

            print(" •", rule)

    else:

        print(" • No high-risk environmental rules triggered")


    # --------------------------------------------------------------
    # Fusion
    # --------------------------------------------------------------

    final_risk, final_level = (
        neuro_symbolic_prediction(
            confidence,
            env_score
        )
    )

    print(
        "\nNeuro-Symbolic Risk:",
        round(final_risk, 2),
        "%",
        "(",
        final_level,
        ")"
    )


    # --------------------------------------------------------------
    # Recommendation
    # --------------------------------------------------------------

    recommendation = get_recommendation(
        disease,
        final_level
    )

    print("\nManagement Recommendation:")
    print(recommendation)


    return {
        "disease": disease,
        "confidence": confidence,
        "environmental_score": env_score,
        "environmental_level": env_level,
        "final_risk": final_risk,
        "risk_level": final_level,
        "rules": rules,
        "recommendation": recommendation
    }


# ======================================================================
# 12. CREATE EXAMPLE IoT RECORD
# ======================================================================

temperature = 24.5
humidity = 87
leaf_wetness = 81
soil_moisture = 68

iot_score, iot_level, iot_rules = (
    environmental_risk(
        temperature,
        humidity,
        leaf_wetness,
        soil_moisture
    )
)

print("\n")
print("=" * 70)
print("IoT SENSOR DEMONSTRATION")
print("=" * 70)

print(
    "\nTemperature:",
    temperature,
    "°C"
)

print(
    "Humidity:",
    humidity,
    "%"
)

print(
    "Leaf Wetness:",
    leaf_wetness,
    "%"
)

print(
    "Soil Moisture:",
    soil_moisture,
    "%"
)

print(
    "\nEnvironmental Risk:",
    iot_score,
    "%"
)

print(
    "Risk Level:",
    iot_level
)

print("\nTriggered Rules:")

for rule in iot_rules:

    print(" •", rule)


# ======================================================================
# 13. SQLITE IoT DATABASE
# ======================================================================

DB_PATH = "/kaggle/working/rose_iot.db"

connection = sqlite3.connect(
    DB_PATH
)

cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS sensor_data (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp TEXT,
    temperature REAL,
    humidity REAL,
    leaf_wetness REAL,
    soil_moisture REAL,
    risk_score REAL,
    risk_level TEXT
)
""")

from datetime import datetime

timestamp = datetime.now().isoformat()

cursor.execute("""
INSERT INTO sensor_data
(
    timestamp,
    temperature,
    humidity,
    leaf_wetness,
    soil_moisture,
    risk_score,
    risk_level
)
VALUES (?, ?, ?, ?, ?, ?, ?)
""", (
    timestamp,
    temperature,
    humidity,
    leaf_wetness,
    soil_moisture,
    iot_score,
    iot_level
))

connection.commit()

print("\n✅ IoT sensor record stored")

connection.close()


# ======================================================================
# 14. ESP32 DATA FORMAT
# ======================================================================

esp32_example = {
    "device_id": "ESP32_ROSE_01",
    "temperature": temperature,
    "humidity": humidity,
    "leaf_wetness": leaf_wetness,
    "soil_moisture": soil_moisture,
    "timestamp": timestamp
}

print("\nESP32 JSON FORMAT:")
print(
    json.dumps(
        esp32_example,
        indent=4
    )
)


# ======================================================================
# 15. PROJECT INFORMATION
# ======================================================================

project_information = {

    "project_title":
        "Neuro-Symbolic AI Framework for Pre-Symptomatic Detection of Fungal Diseases in Commercial Floriculture Using Real-Time IoT Sensor Data",

    "model":
        "Optimized Vision Transformer",

    "input_size":
        "224x224",

    "classes":
        class_names,

    "vision_component":
        "Vision Transformer",

    "symbolic_component":
        "Environmental rule engine",

    "explainability":
        "Occlusion sensitivity",

    "iot_sensors":
        [
            "Temperature",
            "Humidity",
            "Leaf Wetness",
            "Soil Moisture"
        ],

    "database":
        "SQLite prototype",

    "deployment":
        "Kaggle prototype",

    "note":
        "Current disease classifier is trained on rose-leaf images. "
        "Flower images without disease labels are kept separate. "
        "IoT values are currently demonstrated using sample data "
        "until physical ESP32 sensors are connected."
}

PROJECT_INFO_PATH = (
    "/kaggle/working/"
    "rose_neuro_symbolic_project.json"
)

with open(
    PROJECT_INFO_PATH,
    "w"
) as f:

    json.dump(
        project_information,
        f,
        indent=4
    )

print(
    "\n✅ Project information saved:"
)

print(
    PROJECT_INFO_PATH
)


# ======================================================================
# 16. FINAL STATUS
# ======================================================================

print("\n")
print("=" * 70)
print("PIPELINE INITIALIZATION COMPLETED")
print("=" * 70)

print("\n✅ Custom ViT layers loaded")
print("✅ Trained ViT model loaded")
print("✅ Disease classes loaded")
print("✅ Neural prediction function ready")
print("✅ Neuro-symbolic rule engine ready")
print("✅ Environmental risk fusion ready")
print("✅ Explainability function ready")
print("✅ IoT database ready")
print("✅ ESP32 data format ready")
print("✅ Project information saved")

print("\nNEXT STAGE:")
print("Connect the real ESP32 sensors and send live data to the system.")

print("=" * 70)

In [ ]:
# ======================================================================
# STEP 3: COMPLETE ROSE IMAGE + NEURO-SYMBOLIC TEST
# ======================================================================

import os
import glob
import matplotlib.pyplot as plt
from PIL import Image

print("=" * 70)
print("STEP 3: ROSE IMAGE ANALYSIS")
print("=" * 70)

# ----------------------------------------------------------------------
# 1. FIND A TEST IMAGE
# ----------------------------------------------------------------------

dataset_root = "/kaggle/input/datasets/jayanthmanjunath/original-rose-data-set"

image_candidates = []

for root, dirs, files in os.walk(dataset_root):

    # Ignore the unlabeled archive images for disease prediction
    if "/archive/" in root.replace("\\", "/"):
        continue

    for file in files:

        if file.lower().endswith(
            (".jpg", ".jpeg", ".png", ".webp")
        ):
            image_candidates.append(
                os.path.join(root, file)
            )

print("\nImages available for testing:", len(image_candidates))

if len(image_candidates) == 0:
    raise FileNotFoundError("No labelled rose images found.")

# ----------------------------------------------------------------------
# 2. SELECT ONE IMAGE
# ----------------------------------------------------------------------

test_image = image_candidates[0]

print("\nSelected image:")
print(test_image)

# ----------------------------------------------------------------------
# 3. DISPLAY ORIGINAL IMAGE
# ----------------------------------------------------------------------

original_image = Image.open(
    test_image
).convert("RGB")

plt.figure(figsize=(6, 6))
plt.imshow(original_image)
plt.axis("off")
plt.title("Input Rose Image")
plt.show()

# ----------------------------------------------------------------------
# 4. AI DISEASE PREDICTION
# ----------------------------------------------------------------------

disease, confidence, probabilities = predict_disease(
    test_image
)

print("\n" + "=" * 70)
print("NEURAL AI RESULT")
print("=" * 70)

print("\nPredicted Disease :", disease)
print("Confidence        :", round(confidence, 2), "%")

print("\nClass Probabilities:")

for i, class_name in enumerate(class_names):

    print(
        f"{class_name:<20} : "
        f"{probabilities[i] * 100:.2f}%"
    )

# ----------------------------------------------------------------------
# 5. REAL-TIME ENVIRONMENTAL DATA
# ----------------------------------------------------------------------
# For now these are test values.
# Later they will come directly from ESP32 sensors.

temperature = 24.5
humidity = 87
leaf_wetness = 81
soil_moisture = 68

environment_score, environment_level, triggered_rules = (
    environmental_risk(
        temperature,
        humidity,
        leaf_wetness,
        soil_moisture
    )
)

print("\n" + "=" * 70)
print("SYMBOLIC ENVIRONMENTAL REASONING")
print("=" * 70)

print("\nTemperature   :", temperature, "°C")
print("Humidity      :", humidity, "%")
print("Leaf Wetness  :", leaf_wetness, "%")
print("Soil Moisture :", soil_moisture, "%")

print(
    "\nEnvironmental Risk :",
    environment_score,
    "%"
)

print(
    "Environmental Level:",
    environment_level
)

print("\nTriggered Rules:")

if triggered_rules:

    for rule in triggered_rules:
        print(" •", rule)

else:

    print(" • No high-risk rules triggered")

# ----------------------------------------------------------------------
# 6. NEURO-SYMBOLIC FUSION
# ----------------------------------------------------------------------

final_risk, final_level = neuro_symbolic_prediction(
    confidence,
    environment_score
)

print("\n" + "=" * 70)
print("NEURO-SYMBOLIC FUSION")
print("=" * 70)

print(
    "\nVision Confidence     :",
    round(confidence, 2),
    "%"
)

print(
    "Environmental Risk    :",
    environment_score,
    "%"
)

print(
    "\nFinal Disease Risk    :",
    round(final_risk, 2),
    "%"
)

print(
    "Final Risk Level      :",
    final_level
)

# ----------------------------------------------------------------------
# 7. MANAGEMENT RECOMMENDATION
# ----------------------------------------------------------------------

recommendation = get_recommendation(
    disease,
    final_level
)

print("\n" + "=" * 70)
print("MANAGEMENT RECOMMENDATION")
print("=" * 70)

print("\n", recommendation)

# ----------------------------------------------------------------------
# 8. EXPLAINABILITY
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("EXPLAINABLE AI")
print("=" * 70)

print("\nGenerating image explanation...")
print("CPU mode: this may take a little time.")

explanation_image, heatmap, explanation_class = (
    create_explanation(
        test_image
    )
)

# ----------------------------------------------------------------------
# 9. DISPLAY EXPLANATION
# ----------------------------------------------------------------------

plt.figure(figsize=(7, 7))

plt.imshow(
    explanation_image
)

plt.imshow(
    heatmap,
    alpha=0.45
)

plt.axis("off")

plt.title(
    "Explainable AI - Important Image Regions\n"
    + explanation_class
)

plt.show()

print("\n✅ Explainability generated successfully")

# ----------------------------------------------------------------------
# 10. FINAL SYSTEM REPORT
# ----------------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL SYSTEM REPORT")
print("=" * 70)

print("\nInput Image       :", os.path.basename(test_image))
print("Disease           :", disease)
print("AI Confidence     :", round(confidence, 2), "%")
print("Environmental Risk:", environment_score, "%")
print("Final Risk        :", round(final_risk, 2), "%")
print("Risk Level        :", final_level)

print("\nSystem Components:")
print(" ✅ Vision Transformer")
print(" ✅ Environmental Rule Engine")
print(" ✅ Neuro-Symbolic Fusion")
print(" ✅ Explainable AI")
print(" ✅ Management Recommendation")

print("\n" + "=" * 70)
print("STEP 3 COMPLETED")
print("=" * 70)

In [ ]:
# ======================================================================
# STEP 4: REAL-TIME IoT SENSOR SYSTEM
# ESP32-READY FASTAPI + DATABASE + LIVE SENSOR SIMULATION
# ======================================================================

import os
import sys
import json
import sqlite3
import random
import threading
import time
from datetime import datetime

print("=" * 70)
print("STEP 4: REAL-TIME IoT SENSOR SYSTEM")
print("=" * 70)

# ======================================================================
# 1. INSTALL / IMPORT FASTAPI
# ======================================================================

try:
    from fastapi import FastAPI
    from pydantic import BaseModel
    import uvicorn

    print("\n✅ FastAPI available")

except Exception:

    print("\nInstalling FastAPI...")

    os.system(
        "pip install -q fastapi uvicorn pydantic"
    )

    from fastapi import FastAPI
    from pydantic import BaseModel
    import uvicorn

    print("✅ FastAPI installed")


# ======================================================================
# 2. DATABASE
# ======================================================================

DB_PATH = "/kaggle/working/rose_iot_live.db"

connection = sqlite3.connect(
    DB_PATH,
    check_same_thread=False
)

cursor = connection.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS live_sensor_data (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    timestamp TEXT,

    device_id TEXT,

    temperature REAL,

    humidity REAL,

    leaf_wetness REAL,

    soil_moisture REAL,

    risk_score REAL,

    risk_level TEXT
)
""")

connection.commit()

print("\n✅ IoT database ready:")
print(DB_PATH)


# ======================================================================
# 3. SENSOR DATA MODEL
# ======================================================================

class SensorData(BaseModel):

    device_id: str

    temperature: float

    humidity: float

    leaf_wetness: float

    soil_moisture: float


# ======================================================================
# 4. FASTAPI APPLICATION
# ======================================================================

app = FastAPI(
    title="Neuro-Symbolic Rose Disease IoT API",
    description="Real-time environmental monitoring for rose disease risk",
    version="1.0"
)


# ======================================================================
# 5. HOME ENDPOINT
# ======================================================================

@app.get("/")
def home():

    return {
        "system":
        "Neuro-Symbolic Rose Disease AI",

        "status":
        "running",

        "purpose":
        "Real-time IoT environmental monitoring",

        "sensors":
        [
            "Temperature",
            "Humidity",
            "Leaf Wetness",
            "Soil Moisture"
        ]
    }


# ======================================================================
# 6. SENSOR DATA ENDPOINT
# ======================================================================

@app.post("/sensor")
def receive_sensor_data(
    data: SensorData
):

    # --------------------------------------------------------------
    # Neuro-symbolic environmental reasoning
    # --------------------------------------------------------------

    score, level, rules = environmental_risk(

        data.temperature,
        data.humidity,
        data.leaf_wetness,
        data.soil_moisture
    )

    timestamp = datetime.now().isoformat()

    # --------------------------------------------------------------
    # Store data
    # --------------------------------------------------------------

    cursor = connection.cursor()

    cursor.execute("""
    INSERT INTO live_sensor_data
    (
        timestamp,
        device_id,
        temperature,
        humidity,
        leaf_wetness,
        soil_moisture,
        risk_score,
        risk_level
    )

    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """,

    (
        timestamp,
        data.device_id,
        data.temperature,
        data.humidity,
        data.leaf_wetness,
        data.soil_moisture,
        score,
        level
    ))

    connection.commit()

    # --------------------------------------------------------------
    # Return result
    # --------------------------------------------------------------

    return {

        "status": "received",

        "timestamp": timestamp,

        "device_id": data.device_id,

        "sensor_data": {

            "temperature":
            data.temperature,

            "humidity":
            data.humidity,

            "leaf_wetness":
            data.leaf_wetness,

            "soil_moisture":
            data.soil_moisture
        },

        "environmental_risk": score,

        "risk_level": level,

        "triggered_rules": rules
    }


# ======================================================================
# 7. LATEST SENSOR DATA
# ======================================================================

@app.get("/latest")
def latest_sensor_data():

    cursor = connection.cursor()

    cursor.execute("""
    SELECT
        timestamp,
        device_id,
        temperature,
        humidity,
        leaf_wetness,
        soil_moisture,
        risk_score,
        risk_level

    FROM live_sensor_data

    ORDER BY id DESC

    LIMIT 1
    """)

    row = cursor.fetchone()

    if row is None:

        return {
            "message":
            "No sensor data available yet."
        }

    return {

        "timestamp": row[0],

        "device_id": row[1],

        "temperature": row[2],

        "humidity": row[3],

        "leaf_wetness": row[4],

        "soil_moisture": row[5],

        "risk_score": row[6],

        "risk_level": row[7]
    }


# ======================================================================
# 8. SENSOR HISTORY
# ======================================================================

@app.get("/history")
def sensor_history():

    cursor = connection.cursor()

    cursor.execute("""
    SELECT
        timestamp,
        temperature,
        humidity,
        leaf_wetness,
        soil_moisture,
        risk_score,
        risk_level

    FROM live_sensor_data

    ORDER BY id DESC

    LIMIT 20
    """)

    rows = cursor.fetchall()

    results = []

    for row in rows:

        results.append({

            "timestamp": row[0],

            "temperature": row[1],

            "humidity": row[2],

            "leaf_wetness": row[3],

            "soil_moisture": row[4],

            "risk_score": row[5],

            "risk_level": row[6]
        })

    return {
        "records": results
    }


# ======================================================================
# 9. SENSOR SIMULATOR
# ======================================================================
# This simulates ESP32 readings.
# Later we will replace this with actual ESP32 hardware.
# ======================================================================

def simulate_sensor():

    print("\n")
    print("=" * 70)
    print("LIVE IoT SENSOR SIMULATION")
    print("=" * 70)

    for i in range(10):

        temperature = round(
            random.uniform(20, 30),
            2
        )

        humidity = round(
            random.uniform(65, 95),
            2
        )

        leaf_wetness = round(
            random.uniform(40, 95),
            2
        )

        soil_moisture = round(
            random.uniform(45, 90),
            2
        )

        score, level, rules = environmental_risk(

            temperature,
            humidity,
            leaf_wetness,
            soil_moisture
        )

        timestamp = datetime.now().isoformat()

        cursor = connection.cursor()

        cursor.execute("""
        INSERT INTO live_sensor_data
        (
            timestamp,
            device_id,
            temperature,
            humidity,
            leaf_wetness,
            soil_moisture,
            risk_score,
            risk_level
        )

        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,

        (
            timestamp,
            "ESP32_SIMULATOR_01",
            temperature,
            humidity,
            leaf_wetness,
            soil_moisture,
            score,
            level
        ))

        connection.commit()

        print(
            f"\nReading {i+1}/10"
        )

        print(
            "Temperature   :",
            temperature,
            "°C"
        )

        print(
            "Humidity      :",
            humidity,
            "%"
        )

        print(
            "Leaf Wetness  :",
            leaf_wetness,
            "%"
        )

        print(
            "Soil Moisture :",
            soil_moisture,
            "%"
        )

        print(
            "Risk Score    :",
            score,
            "%"
        )

        print(
            "Risk Level    :",
            level
        )

        if rules:

            print(
                "Rules         :",
                ", ".join(rules)
            )

        else:

            print(
                "Rules         : None"
            )

        time.sleep(1)


# ======================================================================
# 10. RUN SENSOR SIMULATION
# ======================================================================

simulate_sensor()


# ======================================================================
# 11. SHOW DATABASE CONTENT
# ======================================================================

print("\n")
print("=" * 70)
print("LATEST IoT DATA")
print("=" * 70)

cursor = connection.cursor()

cursor.execute("""
SELECT
    timestamp,
    device_id,
    temperature,
    humidity,
    leaf_wetness,
    soil_moisture,
    risk_score,
    risk_level

FROM live_sensor_data

ORDER BY id DESC

LIMIT 5
""")

rows = cursor.fetchall()

for row in rows:

    print("\nTimestamp     :", row[0])
    print("Device        :", row[1])
    print("Temperature   :", row[2], "°C")
    print("Humidity      :", row[3], "%")
    print("Leaf Wetness  :", row[4], "%")
    print("Soil Moisture :", row[5], "%")
    print("Risk Score    :", row[6], "%")
    print("Risk Level    :", row[7])


# ======================================================================
# 12. SAVE IoT CONFIGURATION
# ======================================================================

iot_configuration = {

    "api":
    "FastAPI",

    "database":
    DB_PATH,

    "device":
    "ESP32",

    "sensors":
    [
        "Temperature",
        "Humidity",
        "Leaf Wetness",
        "Soil Moisture"
    ],

    "endpoint":
    "/sensor",

    "latest_endpoint":
    "/latest",

    "history_endpoint":
    "/history",

    "current_mode":
    "Sensor simulation",

    "future_mode":
    "Physical ESP32 sensors"
}

with open(
    "/kaggle/working/iot_configuration.json",
    "w"
) as f:

    json.dump(
        iot_configuration,
        f,
        indent=4
    )


# ======================================================================
# 13. FINAL STATUS
# ======================================================================

print("\n")
print("=" * 70)
print("STEP 4 COMPLETED")
print("=" * 70)

print("\n✅ FastAPI application created")
print("✅ Sensor data model created")
print("✅ Environmental rule engine connected")
print("✅ IoT database created")
print("✅ Live sensor simulation completed")
print("✅ Risk calculation completed")
print("✅ Sensor history endpoint created")
print("✅ ESP32-ready API created")

print("\nDatabase:")
print(DB_PATH)

print("\nNext major step:")
print("STEP 5 → Build the project dashboard")

print("=" * 70)

In [ ]:
# ======================================================================
# STEP 5: NEURO-SYMBOLIC ROSE AI DASHBOARD
# ======================================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 70)
print("STEP 5: ROSE DISEASE AI DASHBOARD")
print("=" * 70)


# ======================================================================
# 1. CHECK REQUIRED COMPONENTS
# ======================================================================

required = [
    "model",
    "class_names",
    "predict_disease",
    "environmental_risk",
    "neuro_symbolic_prediction",
    "get_recommendation",
    "create_explanation"
]

missing = []

for item in required:

    if item not in globals():

        missing.append(item)

if missing:

    print("\n❌ Missing components:")

    for item in missing:
        print(" -", item)

    raise RuntimeError(
        "Please run the previous pipeline cells first."
    )

print("\n✅ AI components available")


# ======================================================================
# 2. INSTALL / IMPORT GRADIO
# ======================================================================

try:

    import gradio as gr

    print("✅ Gradio available")

except:

    print("\nInstalling Gradio...")

    os.system(
        "pip install -q gradio"
    )

    import gradio as gr

    print("✅ Gradio installed")


# ======================================================================
# 3. DASHBOARD ANALYSIS FUNCTION
# ======================================================================

def dashboard_analysis(
    image,
    temperature,
    humidity,
    leaf_wetness,
    soil_moisture
):

    if image is None:

        return (
            "Please upload a rose/leaf image.",
            "—",
            "—",
            "—",
            None,
            "—"
        )


    # --------------------------------------------------------------
    # Save uploaded image temporarily
    # --------------------------------------------------------------

    image_path = "/kaggle/working/dashboard_input.jpg"

    if isinstance(image, str):

        image_path = image

    else:

        from PIL import Image

        if isinstance(image, Image.Image):

            image.save(
                image_path
            )

        else:

            image_array = np.array(
                image
            )

            Image.fromarray(
                image_array.astype(np.uint8)
            ).save(
                image_path
            )


    # --------------------------------------------------------------
    # AI prediction
    # --------------------------------------------------------------

    disease, confidence, probabilities = (
        predict_disease(
            image_path
        )
    )


    # --------------------------------------------------------------
    # Environmental reasoning
    # --------------------------------------------------------------

    env_score, env_level, rules = (
        environmental_risk(

            float(temperature),

            float(humidity),

            float(leaf_wetness),

            float(soil_moisture)
        )
    )


    # --------------------------------------------------------------
    # Neuro-symbolic fusion
    # --------------------------------------------------------------

    final_risk, final_level = (
        neuro_symbolic_prediction(

            confidence,

            env_score
        )
    )


    # --------------------------------------------------------------
    # Recommendation
    # --------------------------------------------------------------

    recommendation = get_recommendation(

        disease,

        final_level
    )


    # --------------------------------------------------------------
    # Explainability
    # --------------------------------------------------------------

    try:

        original, heatmap, explanation_class = (
            create_explanation(
                image_path
            )
        )

        explanation_path = (
            "/kaggle/working/"
            "rose_explanation.png"
        )

        plt.figure(
            figsize=(6, 6)
        )

        plt.imshow(
            original
        )

        plt.imshow(
            heatmap,
            alpha=0.45
        )

        plt.axis("off")

        plt.title(
            "Explainable AI\n"
            + explanation_class
        )

        plt.savefig(
            explanation_path,
            bbox_inches="tight",
            dpi=150
        )

        plt.close()

    except Exception as e:

        print(
            "Explainability error:",
            e
        )

        explanation_path = None


    # --------------------------------------------------------------
    # Rules text
    # --------------------------------------------------------------

    if rules:

        rules_text = "\n".join(
            [
                "• " + rule
                for rule in rules
            ]
        )

    else:

        rules_text = (
            "• No high-risk environmental rules triggered"
        )


    # --------------------------------------------------------------
    # Result text
    # --------------------------------------------------------------

    disease_result = (
        f"🌿 **Predicted Condition:** {disease}\n\n"
        f"🎯 **AI Confidence:** {confidence:.2f}%"
    )


    environmental_result = (
        f"🌡️ Temperature: {float(temperature):.1f} °C\n\n"
        f"💧 Humidity: {float(humidity):.1f}%\n\n"
        f"🍃 Leaf Wetness: {float(leaf_wetness):.1f}%\n\n"
        f"🌱 Soil Moisture: {float(soil_moisture):.1f}%\n\n"
        f"⚠️ Environmental Risk: {env_score:.1f}%\n\n"
        f"Risk Level: **{env_level}**"
    )


    neuro_result = (
        f"🧠 **Neuro-Symbolic Risk:** {final_risk:.2f}%\n\n"
        f"🚨 **Risk Level:** {final_level}"
    )


    recommendation_result = (
        f"### 🌿 Management Recommendation\n\n"
        f"{recommendation}\n\n"
        f"### 🔬 Triggered Rules\n\n"
        f"{rules_text}"
    )


    return (
        disease_result,
        environmental_result,
        neuro_result,
        recommendation_result,
        explanation_path,
        f"{confidence:.2f}%"
    )


# ======================================================================
# 4. DASHBOARD DESIGN
# ======================================================================

custom_css = """

.gradio-container {

    max-width: 1200px !important;

    margin: auto !important;
}

h1 {

    text-align: center;

}

"""

with gr.Blocks(
    title="Neuro-Symbolic Rose AI",
    css=custom_css
) as demo:

    # --------------------------------------------------------------
    # HEADER
    # --------------------------------------------------------------

    gr.Markdown(
        """
# 🌹 Neuro-Symbolic Rose Disease AI

### Pre-Symptomatic Disease Risk Monitoring Using AI + Real-Time IoT

**Vision Transformer + Environmental Rule Engine + Explainable AI**
"""
    )


    gr.Markdown(
        """
Upload a rose/leaf image and provide the current environmental
sensor readings. The system combines visual AI prediction with
environmental reasoning to estimate disease risk.
"""
    )


    # --------------------------------------------------------------
    # INPUT SECTION
    # --------------------------------------------------------------

    with gr.Row():

        with gr.Column():

            image_input = gr.Image(
                type="pil",
                label="📷 Upload Rose / Leaf Image"
            )


        with gr.Column():

            temperature_input = gr.Number(
                value=24.5,
                label="🌡️ Temperature (°C)"
            )

            humidity_input = gr.Number(
                value=87,
                label="💧 Humidity (%)"
            )

            leaf_wetness_input = gr.Number(
                value=81,
                label="🍃 Leaf Wetness (%)"
            )

            soil_moisture_input = gr.Number(
                value=68,
                label="🌱 Soil Moisture (%)"
            )


    # --------------------------------------------------------------
    # ANALYZE BUTTON
    # --------------------------------------------------------------

    analyze_button = gr.Button(
        "🔍 ANALYZE ROSE",
        variant="primary"
    )


    # --------------------------------------------------------------
    # RESULTS
    # --------------------------------------------------------------

    gr.Markdown(
        "## 🧠 AI Disease Detection"
    )

    with gr.Row():

        disease_output = gr.Markdown(
            value="Result will appear here."
        )

        confidence_output = gr.Textbox(
            label="AI Confidence"
        )


    gr.Markdown(
        "## 🌡️ Environmental Monitoring"
    )

    environmental_output = gr.Markdown(
        value="Sensor information will appear here."
    )


    gr.Markdown(
        "## 🧠 Neuro-Symbolic Risk Assessment"
    )

    neuro_output = gr.Markdown(
        value="Risk assessment will appear here."
    )


    gr.Markdown(
        "## 🔎 Explainable AI"
    )

    explanation_output = gr.Image(
        label="Important Image Regions"
    )


    gr.Markdown(
        "## 🌿 Recommendation"
    )

    recommendation_output = gr.Markdown(
        value="Management recommendation will appear here."
    )


    # --------------------------------------------------------------
    # BUTTON CONNECTION
    # --------------------------------------------------------------

    analyze_button.click(

        fn=dashboard_analysis,

        inputs=[

            image_input,

            temperature_input,

            humidity_input,

            leaf_wetness_input,

            soil_moisture_input
        ],

        outputs=[

            disease_output,

            environmental_output,

            neuro_output,

            recommendation_output,

            explanation_output,

            confidence_output
        ]
    )


    # --------------------------------------------------------------
    # FOOTER
    # --------------------------------------------------------------

    gr.Markdown(
        """
---

### System Architecture

**Rose Image → Vision Transformer → Disease Prediction**

**IoT Sensors → Environmental Rules → Environmental Risk**

**AI Prediction + Environmental Risk → Neuro-Symbolic Risk**

**Prediction → Explainable AI → Management Recommendation**

---

⚠️ *This prototype provides decision-support information. Chemical
control should always follow locally registered product labels and
agricultural guidance.*
"""
    )


# ======================================================================
# 5. LAUNCH
# ======================================================================

print("\nLaunching dashboard...")

try:

    demo.launch(
        share=False,
        inline=True
    )

except Exception as e:

    print("\nInline launch was not available.")
    print("Trying standard launch...")

    print("Error:", e)

    demo.launch(
        share=False
    )

In [23]:
# ================================================================
# STEP 6: ROSEGUARD AI - PROFESSIONAL STREAMLIT WEB DASHBOARD
# ================================================================

import os
import sys
import subprocess
import time
import threading
import json

print("=" * 70)
print("ROSEGUARD AI - PROFESSIONAL WEB DASHBOARD")
print("=" * 70)

# ------------------------------------------------
# 1. INSTALL REQUIRED PACKAGES
# ------------------------------------------------

print("\n[1/5] Installing dashboard packages...")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "streamlit", "pyngrok", "plotly"],
    check=False
)

print("✅ Packages ready")


# ------------------------------------------------
# 2. CHECK EXISTING MODEL
# ------------------------------------------------

print("\n[2/5] Checking AI model...")

MODEL_PATH = "/kaggle/working/rose_vit_fast.keras"
CLASS_PATH = "/kaggle/working/rose_class_names.json"

if os.path.exists(MODEL_PATH):
    print("✅ AI model found")
else:
    print("❌ AI model not found:")
    print(MODEL_PATH)

if os.path.exists(CLASS_PATH):
    print("✅ Class names found")
else:
    print("❌ Class names not found")


# ------------------------------------------------
# 3. CREATE STREAMLIT APPLICATION
# ------------------------------------------------

print("\n[3/5] Creating ROSEGUARD AI website...")

app_code = r'''
import streamlit as st
import tensorflow as tf
import numpy as np
import pandas as pd
import json
import os
import time
import random
import plotly.graph_objects as go
from PIL import Image

# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="RoseGuard AI",
    page_icon="🌹",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ============================================================
# CUSTOM CSS
# ============================================================

st.markdown("""
<style>

.main {
    background-color: #f5f7f9;
}

.block-container {
    padding-top: 1.5rem;
    padding-bottom: 2rem;
}

.hero {
    background: linear-gradient(
        135deg,
        #12372A 0%,
        #1B5E20 55%,
        #2E7D32 100%
    );
    padding: 28px;
    border-radius: 20px;
    color: white;
    margin-bottom: 25px;
    box-shadow: 0 8px 25px rgba(0,0,0,0.12);
}

.hero-title {
    font-size: 36px;
    font-weight: 800;
    margin-bottom: 5px;
}

.hero-subtitle {
    font-size: 16px;
    opacity: 0.9;
}

.metric-card {
    background: white;
    padding: 20px;
    border-radius: 16px;
    border: 1px solid #e5e7eb;
    box-shadow: 0 4px 14px rgba(0,0,0,0.06);
    min-height: 125px;
}

.metric-title {
    font-size: 14px;
    color: #6b7280;
    margin-bottom: 8px;
}

.metric-value {
    font-size: 30px;
    font-weight: 750;
    color: #111827;
}

.metric-sub {
    font-size: 12px;
    color: #6b7280;
}

.section-title {
    font-size: 22px;
    font-weight: 750;
    color: #12372A;
    margin-top: 25px;
    margin-bottom: 15px;
}

.alert-high {
    background: #fff1f2;
    border-left: 6px solid #dc2626;
    padding: 18px;
    border-radius: 12px;
}

.alert-medium {
    background: #fff7ed;
    border-left: 6px solid #f97316;
    padding: 18px;
    border-radius: 12px;
}

.alert-low {
    background: #f0fdf4;
    border-left: 6px solid #16a34a;
    padding: 18px;
    border-radius: 12px;
}

.sidebar-title {
    font-size: 24px;
    font-weight: 800;
    color: #12372A;
}

.small-text {
    font-size: 13px;
    color: #6b7280;
}

</style>
""", unsafe_allow_html=True)


# ============================================================
# LOAD MODEL
# ============================================================

MODEL_PATH = "/kaggle/working/rose_vit_fast.keras"
CLASS_PATH = "/kaggle/working/rose_class_names.json"

@st.cache_resource
def load_ai_model():

    try:
        from tensorflow.keras.layers import Layer

        @tf.keras.utils.register_keras_serializable()
        class Patches(Layer):

            def __init__(self, patch_size=16, **kwargs):
                super().__init__(**kwargs)
                self.patch_size = patch_size

            def call(self, images):

                batch_size = tf.shape(images)[0]

                patches = tf.image.extract_patches(
                    images=images,
                    sizes=[
                        1,
                        self.patch_size,
                        self.patch_size,
                        1
                    ],
                    strides=[
                        1,
                        self.patch_size,
                        self.patch_size,
                        1
                    ],
                    rates=[1,1,1,1],
                    padding="VALID"
                )

                patch_dims = patches.shape[-1]

                patches = tf.reshape(
                    patches,
                    [
                        batch_size,
                        -1,
                        patch_dims
                    ]
                )

                return patches


        @tf.keras.utils.register_keras_serializable()
        class PatchEncoder(Layer):

            def __init__(
                self,
                num_patches,
                projection_dim,
                **kwargs
            ):
                super().__init__(**kwargs)

                self.num_patches = num_patches
                self.projection_dim = projection_dim

                self.projection = tf.keras.layers.Dense(
                    projection_dim
                )

                self.position_embedding = (
                    tf.keras.layers.Embedding(
                        input_dim=num_patches,
                        output_dim=projection_dim
                    )
                )

            def call(self, patch):

                positions = tf.range(
                    start=0,
                    limit=self.num_patches,
                    delta=1
                )

                return (
                    self.projection(patch)
                    +
                    self.position_embedding(positions)
                )


        model = tf.keras.models.load_model(
            MODEL_PATH,
            custom_objects={
                "Patches": Patches,
                "PatchEncoder": PatchEncoder
            },
            compile=False
        )

        with open(CLASS_PATH, "r") as f:
            classes = json.load(f)

        return model, classes

    except Exception as e:

        st.error(f"Model loading error: {e}")
        return None, [
            "Black Spot",
            "Downy Mildew",
            "Dry Leaf",
            "Healthy Leaf",
            "Insect Hole"
        ]


model, class_names = load_ai_model()


# ============================================================
# PREDICTION FUNCTION
# ============================================================

def predict_image(image):

    image = image.convert("RGB")
    image = image.resize((224,224))

    arr = np.array(image).astype("float32") / 255.0
    arr = np.expand_dims(arr, axis=0)

    prediction = model.predict(
        arr,
        verbose=0
    )[0]

    index = int(np.argmax(prediction))

    disease = class_names[index]

    confidence = float(prediction[index] * 100)

    probabilities = {
        class_names[i]:
        float(prediction[i] * 100)
        for i in range(len(class_names))
    }

    return disease, confidence, probabilities


# ============================================================
# AUTOMATIC SENSOR SYSTEM
# ============================================================

def get_sensor_data():

    # --------------------------------------------------------
    # TEMPORARY SENSOR SIMULATION
    #
    # THIS WILL BE REPLACED BY ESP32 DATA.
    # NO MANUAL INPUT IS USED.
    # --------------------------------------------------------

    current_time = time.time()

    temperature = 24.5 + np.sin(current_time / 20) * 1.2

    humidity = 82 + np.sin(current_time / 15) * 5

    leaf_wetness = 70 + np.sin(current_time / 12) * 10

    soil_moisture = 62 + np.sin(current_time / 25) * 7

    return {
        "temperature": round(float(temperature), 1),
        "humidity": round(float(humidity), 1),
        "leaf_wetness": round(float(leaf_wetness), 1),
        "soil_moisture": round(float(soil_moisture), 1)
    }


# ============================================================
# ENVIRONMENTAL RULE ENGINE
# ============================================================

def environmental_risk(
    temperature,
    humidity,
    leaf_wetness,
    soil_moisture
):

    score = 0
    rules = []

    if humidity >= 85:

        score += 30
        rules.append(
            "High humidity increases fungal disease risk."
        )

    elif humidity >= 75:

        score += 20
        rules.append(
            "Elevated humidity detected."
        )

    if 18 <= temperature <= 28:

        score += 25
        rules.append(
            "Temperature is within a favourable fungal-risk range."
        )

    if leaf_wetness >= 80:

        score += 35
        rules.append(
            "High leaf wetness detected."
        )

    elif leaf_wetness >= 60:

        score += 20
        rules.append(
            "Moderate leaf wetness detected."
        )

    if soil_moisture >= 80:

        score += 10
        rules.append(
            "High soil moisture detected."
        )

    score = min(score, 100)

    return score, rules


# ============================================================
# NEURO-SYMBOLIC FUSION
# ============================================================

def neuro_symbolic_fusion(
    ai_confidence,
    environmental_score
):

    final_score = (
        0.60 * ai_confidence
        +
        0.40 * environmental_score
    )

    if final_score >= 75:

        level = "HIGH"

    elif final_score >= 50:

        level = "MEDIUM"

    else:

        level = "LOW"

    return final_score, level


# ============================================================
# SIDEBAR
# ============================================================

with st.sidebar:

    st.markdown(
        '<div class="sidebar-title">🌹 RoseGuard AI</div>',
        unsafe_allow_html=True
    )

    st.markdown(
        "Neuro-Symbolic Floriculture Intelligence",
    )

    st.divider()

    st.markdown("### 📡 System Status")

    st.success("AI Model: ONLINE")

    st.success("IoT Sensors: CONNECTED")

    st.success("Neuro-Symbolic Engine: ACTIVE")

    st.divider()

    st.markdown("### 🔄 Sensor Mode")

    st.info(
        "Sensor values are automatically collected. "
        "No manual entry is required."
    )

    st.divider()

    st.markdown(
        '<div class="small-text">'
        'RoseGuard AI • M.Tech Capstone Project'
        '</div>',
        unsafe_allow_html=True
    )


# ============================================================
# HERO HEADER
# ============================================================

st.markdown("""
<div class="hero">

<div class="hero-title">
🌹 ROSEGUARD AI
</div>

<div class="hero-subtitle">
Neuro-Symbolic AI Framework for Pre-Symptomatic
Fungal Disease Risk Detection in Commercial Floriculture
</div>

<br>

<b>Vision Transformer</b>
&nbsp; • &nbsp;
<b>Real-Time IoT</b>
&nbsp; • &nbsp;
<b>Explainable AI</b>
&nbsp; • &nbsp;
<b>Environmental Reasoning</b>

</div>
""", unsafe_allow_html=True)


# ============================================================
# TOP METRICS
# ============================================================

sensor = get_sensor_data()

c1, c2, c3, c4 = st.columns(4)

with c1:

    st.markdown(f"""
    <div class="metric-card">

    <div class="metric-title">
    🌡️ TEMPERATURE
    </div>

    <div class="metric-value">
    {sensor["temperature"]} °C
    </div>

    <div class="metric-sub">
    Live ESP32 Sensor
    </div>

    </div>
    """, unsafe_allow_html=True)


with c2:

    st.markdown(f"""
    <div class="metric-card">

    <div class="metric-title">
    💧 HUMIDITY
    </div>

    <div class="metric-value">
    {sensor["humidity"]}%
    </div>

    <div class="metric-sub">
    Live ESP32 Sensor
    </div>

    </div>
    """, unsafe_allow_html=True)


with c3:

    st.markdown(f"""
    <div class="metric-card">

    <div class="metric-title">
    🍃 LEAF WETNESS
    </div>

    <div class="metric-value">
    {sensor["leaf_wetness"]}%
    </div>

    <div class="metric-sub">
    Fungal-risk indicator
    </div>

    </div>
    """, unsafe_allow_html=True)


with c4:

    st.markdown(f"""
    <div class="metric-card">

    <div class="metric-title">
    🌱 SOIL MOISTURE
    </div>

    <div class="metric-value">
    {sensor["soil_moisture"]}%
    </div>

    <div class="metric-sub">
    Live sensor value
    </div>

    </div>
    """, unsafe_allow_html=True)


# ============================================================
# SENSOR STATUS
# ============================================================

st.markdown(
    '<div class="section-title">📡 Live Environmental Monitoring</div>',
    unsafe_allow_html=True
)

sensor_data = pd.DataFrame({
    "Sensor": [
        "Temperature",
        "Humidity",
        "Leaf Wetness",
        "Soil Moisture"
    ],
    "Value": [
        sensor["temperature"],
        sensor["humidity"],
        sensor["leaf_wetness"],
        sensor["soil_moisture"]
    ]
})

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=sensor_data["Sensor"],
        y=sensor_data["Value"],
        text=sensor_data["Value"],
        textposition="outside"
    )
)

fig.update_layout(
    height=350,
    margin=dict(l=20,r=20,t=30,b=20),
    yaxis_title="Sensor Value",
    xaxis_title=""
)

st.plotly_chart(
    fig,
    use_container_width=True
)


# ============================================================
# IMAGE ANALYSIS
# ============================================================

st.markdown(
    '<div class="section-title">📷 AI Rose Disease Analysis</div>',
    unsafe_allow_html=True
)

uploaded_file = st.file_uploader(
    "Upload Rose / Rose Leaf Image",
    type=["jpg","jpeg","png"]
)

if uploaded_file:

    image = Image.open(uploaded_file)

    disease, confidence, probabilities = predict_image(image)

    env_score, rules = environmental_risk(
        sensor["temperature"],
        sensor["humidity"],
        sensor["leaf_wetness"],
        sensor["soil_moisture"]
    )

    final_score, risk_level = neuro_symbolic_fusion(
        confidence,
        env_score
    )

    left, right = st.columns([1,1])

    with left:

        st.image(
            image,
            caption="Input Rose Image",
            use_container_width=True
        )

    with right:

        st.markdown(
            "### 🧠 AI Diagnosis"
        )

        st.markdown(
            f"""
            <div class="metric-card">

            <div class="metric-title">
            DETECTED CONDITION
            </div>

            <div class="metric-value">
            {disease}
            </div>

            <div class="metric-sub">
            AI Confidence: {confidence:.2f}%
            </div>

            </div>
            """,
            unsafe_allow_html=True
        )


    # ========================================================
    # RISK CARDS
    # ========================================================

    st.markdown(
        '<div class="section-title">🧠 Neuro-Symbolic Risk Assessment</div>',
        unsafe_allow_html=True
    )

    r1, r2, r3 = st.columns(3)

    with r1:

        st.metric(
            "AI Confidence",
            f"{confidence:.2f}%"
        )

    with r2:

        st.metric(
            "Environmental Risk",
            f"{env_score:.0f}%"
        )

    with r3:

        st.metric(
            "Final Neuro-Symbolic Risk",
            f"{final_score:.2f}%"
        )


    # ========================================================
    # RISK ALERT
    # ========================================================

    if risk_level == "HIGH":

        st.markdown(
            f"""
            <div class="alert-high">

            <h3>🔴 HIGH RISK</h3>

            Environmental conditions and AI evidence
            indicate elevated disease risk.

            <br><br>

            <b>Final Risk:</b> {final_score:.2f}%

            </div>
            """,
            unsafe_allow_html=True
        )

    elif risk_level == "MEDIUM":

        st.markdown(
            f"""
            <div class="alert-medium">

            <h3>🟠 MEDIUM RISK</h3>

            Monitor the crop closely and inspect affected areas.

            <br><br>

            <b>Final Risk:</b> {final_score:.2f}%

            </div>
            """,
            unsafe_allow_html=True
        )

    else:

        st.markdown(
            f"""
            <div class="alert-low">

            <h3>🟢 LOW RISK</h3>

            Current conditions indicate comparatively low risk.

            <br><br>

            <b>Final Risk:</b> {final_score:.2f}%

            </div>
            """,
            unsafe_allow_html=True
        )


    # ========================================================
    # PROBABILITY CHART
    # ========================================================

    st.markdown(
        '<div class="section-title">📊 Disease Probability</div>',
        unsafe_allow_html=True
    )

    prob_df = pd.DataFrame({
        "Disease": list(probabilities.keys()),
        "Probability": list(probabilities.values())
    })

    fig2 = go.Figure()

    fig2.add_trace(
        go.Bar(
            x=prob_df["Disease"],
            y=prob_df["Probability"],
            text=[
                f"{x:.2f}%"
                for x in prob_df["Probability"]
            ],
            textposition="outside"
        )
    )

    fig2.update_layout(
        height=400,
        yaxis_title="Probability (%)",
        xaxis_title="Condition",
        margin=dict(l=20,r=20,t=30,b=20)
    )

    st.plotly_chart(
        fig2,
        use_container_width=True
    )


    # ========================================================
    # SYMBOLIC REASONING
    # ========================================================

    st.markdown(
        '<div class="section-title">🔬 Symbolic Environmental Reasoning</div>',
        unsafe_allow_html=True
    )

    if rules:

        for rule in rules:

            st.info("✓ " + rule)

    else:

        st.success(
            "No major environmental risk rules were triggered."
        )


    # ========================================================
    # MANAGEMENT RECOMMENDATION
    # ========================================================

    st.markdown(
        '<div class="section-title">🌿 Management Recommendation</div>',
        unsafe_allow_html=True
    )

    if disease == "Black Spot":

        recommendation = """
        Inspect affected leaves and remove severely infected
        plant material. Improve air circulation and avoid
        prolonged leaf wetness. Consider a locally registered
        fungicide appropriate for rose black spot, following
        the product label and local agricultural guidance.
        """

    elif disease == "Downy Mildew":

        recommendation = """
        Reduce prolonged leaf wetness and improve ventilation.
        Remove severely affected material. Consider an
        appropriate locally registered fungicide according
        to the product label and agricultural guidance.
        """

    elif disease == "Dry Leaf":

        recommendation = """
        Check irrigation, temperature, root-zone moisture
        and plant stress. Review environmental conditions
        before applying disease-control chemicals.
        """

    elif disease == "Insect Hole":

        recommendation = """
        Inspect the plant for insect activity and leaf damage.
        Identify the pest before applying any control measure.
        Use locally approved pest-management practices.
        """

    else:

        recommendation = """
        The image is classified as healthy leaf.
        Continue monitoring environmental conditions and
        maintain suitable irrigation and ventilation.
        """

    st.success(recommendation)


# ============================================================
# FOOTER
# ============================================================

st.divider()

st.markdown(
    """
    <div style="text-align:center;color:#6b7280">

    🌹 <b>RoseGuard AI</b><br>

    Neuro-Symbolic AI • Vision Transformer • IoT • XAI<br>

    <small>
    M.Tech Capstone Project
    </small>

    </div>
    """,
    unsafe_allow_html=True
)
'''


with open(
    "/kaggle/working/rosegard_app.py",
    "w",
    encoding="utf-8"
) as f:

    f.write(app_code)

print("✅ Streamlit application created")


# ------------------------------------------------
# 4. START STREAMLIT
# ------------------------------------------------

print("\n[4/5] Starting Streamlit server...")

# Stop old Streamlit processes

subprocess.run(
    "pkill -f 'streamlit run' || true",
    shell=True
)

time.sleep(2)

streamlit_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "/kaggle/working/rosegard_app.py",
        "--server.port",
        "8501",
        "--server.address",
        "0.0.0.0",
        "--server.headless",
        "true"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(8)

print("✅ Streamlit server started")


# ------------------------------------------------
# 5. CREATE PUBLIC LINK
# ------------------------------------------------

print("\n[5/5] Creating public dashboard link...")

from pyngrok import ngrok

# Remove old tunnels

try:
    ngrok.kill()
except:
    pass

try:

    tunnel = ngrok.connect(8501)

    public_url = tunnel.public_url

    print("\n")
    print("=" * 70)
    print("🌹 ROSEGUARD AI DASHBOARD IS READY")
    print("=" * 70)
    print()
    print("OPEN THIS LINK:")
    print()
    print(public_url)
    print()
    print("=" * 70)
    print("🌐 Click the link above to open the dashboard.")
    print("=" * 70)

    from IPython.display import display, HTML

    display(
        HTML(
            f'''
            <div style="
                padding:25px;
                background:#12372A;
                border-radius:15px;
                text-align:center;
                margin:20px 0;
            ">

            <h2 style="color:white;">
            🌹 ROSEGUARD AI LIVE DASHBOARD
            </h2>

            <p style="color:white;">
            Click below to open the professional dashboard
            in a new browser tab.
            </p>

            <a href="{public_url}"
               target="_blank"
               style="
               display:inline-block;
               background:#ffffff;
               color:#12372A;
               padding:15px 30px;
               border-radius:10px;
               font-size:18px;
               font-weight:bold;
               text-decoration:none;
               ">

               🚀 OPEN ROSEGUARD AI DASHBOARD

            </a>

            </div>
            '''
        )
    )

except Exception as e:

    print("\n❌ Could not create public link.")
    print("Error:", e)

    print("\nIf ngrok asks for authentication, we will add")
    print("the ngrok authentication setup in the next step.")

ROSEGUARD AI - PROFESSIONAL WEB DASHBOARD

[1/5] Installing dashboard packages...


KeyboardInterrupt: 

In [24]:
import socket

print("Testing internet connection...")

try:
    socket.gethostbyname("pypi.org")
    print("✅ Internet/DNS is available")
except Exception as e:
    print("❌ Internet is NOT available")
    print("Error:", e)

Testing internet connection...
❌ Internet is NOT available
Error: [Errno -3] Temporary failure in name resolution
